# 가위바위보 파인튜닝 (내 손 데이터 추가) - Google Colab

먼저 로컬에서 `merge_own_data.py` 를 실행해서 직접 찍은 데이터를
Roboflow 데이터셋 폴더(`Paper-Rock-Scissors-2`)에 합친 뒤,
그 폴더 전체를 압축(zip)해서 여기에 업로드하세요.

```bash
# 로컬(Jetson)에서 실행
python merge_own_data.py
zip -r merged_dataset.zip Paper-Rock-Scissors-2
```

**실행 순서**
1. `런타임 > 런타임 유형 변경` 에서 GPU 설정
2. 아래 셀 순서대로 실행하면서, 안내되는 시점에 파일 업로드


In [ ]:
# 1. 필요한 패키지 설치
!pip install -q ultralytics


## 2. 합쳐진 데이터셋(zip) 업로드

아래 셀 실행하면 파일 선택 창이 뜹니다. `merged_dataset.zip` 을 선택하세요.


In [ ]:
from google.colab import files
uploaded = files.upload()  # merged_dataset.zip 선택

import zipfile
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

print("압축 해제 완료")
!ls


## 3. 이전에 학습한 best.pt 업로드

1차 학습에서 다운받은 `best.pt` 파일을 여기서 업로드합니다.
(이 가중치를 시작점으로 이어서 학습 = 파인튜닝)


In [ ]:
uploaded2 = files.upload()  # best.pt 선택
pt_name = list(uploaded2.keys())[0]
print(pt_name)


## 4. 파인튜닝 실행

`data.yaml` 경로는 1번 셀에서 압축 해제된 폴더명에 맞게 필요시 수정하세요.
(폴더명이 `Paper-Rock-Scissors-2` 라고 가정)


In [ ]:
from ultralytics import YOLO

model = YOLO(pt_name)  # 처음부터가 아니라 기존 best.pt에서 이어서 학습

model.train(
    data="Paper-Rock-Scissors-2/data.yaml",
    imgsz=640,
    epochs=30,          # 이미 한번 학습된 가중치라 epoch은 적게 잡아도 됨
    batch=16,
    device=0,
    project="runs_rps_finetune",
    name="rps_yolo11n_v2",
    exist_ok=True,
)

metrics = model.val()
print(metrics)


## 5. 새 best.pt 다운로드


In [ ]:
import glob
found = glob.glob("**/rps_yolo11n_v2/weights/best.pt", recursive=True)
print(found)

from google.colab import files
files.download(found[0])
